# Pulse-LM: fine-tune a small open model as the Qoneqt Pulse planner

This notebook fine-tunes **Qwen3-4B-Instruct** (Apache 2.0) with LoRA so that one call turns an input into a **script + shot plan** in JSON.

- Training data: synthetic examples in `data/pulse-lm/` of the repo. Gemini wrote them as the teacher. Our validators, duration governor and Script Critic filtered them (kept only scores of 8 or more).
- No real posts or personal data are used.

**How to run**
1. Menu: Runtime -> Change runtime type -> **T4 GPU** -> Save.
2. Menu: Runtime -> **Run all**.
3. When the token box appears near the end, paste your Hugging Face **write** token and press Enter.
4. Copy the model link and the results block printed at the very end and send them back.

Expected time: about 30 to 50 minutes. Most of it is the GGUF export at the end.

In [ ]:
%%capture
# Same versions as the official Unsloth Qwen3 (4B) Instruct notebook
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# ---- Settings ----
DATA_BASE   = "https://raw.githubusercontent.com/Vikashpatanvadiya/qoneqt/main/data/pulse-lm"
BASE_MODEL  = "unsloth/Qwen3-4B-Instruct-2507"
HF_MODEL    = "pulse-lm-qwen3-4b"   # the repo name created under your Hugging Face account
MAX_SEQ_LEN = 3072                  # system prompt + input + JSON answer is about 1,800 tokens
EPOCHS      = 3

In [ ]:
from datasets import load_dataset

data = load_dataset("json", data_files={
    "train": f"{DATA_BASE}/train.jsonl",
    "val":   f"{DATA_BASE}/val.jsonl",
    "test":  f"{DATA_BASE}/test.jsonl",
})
print(data)
assert len(data["train"]) > 50, "Training file looks empty. Is the repo pushed and public?"

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
    full_finetuning = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "qwen3-instruct")

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize = False, add_generation_prompt = False) for m in batch["messages"]]}

train_ds = data["train"].map(to_text, batched = True)
val_ds   = data["val"].map(to_text, batched = True)

lengths = [len(tokenizer(t).input_ids) for t in train_ds["text"]]
print(f"Token length: max {max(lengths)}, average {sum(lengths)//len(lengths)}. Limit is {MAX_SEQ_LEN}.")
assert max(lengths) <= MAX_SEQ_LEN, "An example is longer than MAX_SEQ_LEN. Raise MAX_SEQ_LEN in the settings cell."

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = EPOCHS,
        warmup_steps = 5,
        learning_rate = 2e-4,
        logging_steps = 5,
        eval_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# Learn only from the assistant's JSON answer, not from the prompt.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
import time
t0 = time.time()
stats = trainer.train()
train_minutes = (time.time() - t0) / 60
eval_loss = trainer.evaluate()["eval_loss"]
print(f"Training took {train_minutes:.1f} min. Train loss {stats.training_loss:.3f}, validation loss {eval_loss:.3f}")

In [ ]:
# Quick check on the validation inputs: does the model return valid JSON with the right keys?
import json
FastLanguageModel.for_inference(model)

def plan(messages, max_new_tokens = 1500):
    text = tokenizer.apply_chat_template(messages[:2], tokenize = False, add_generation_prompt = True)
    inputs = tokenizer(text, return_tensors = "pt").to("cuda")
    t = time.time()
    out = model.generate(**inputs, max_new_tokens = max_new_tokens, do_sample = False)
    new_tokens = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens = True), len(new_tokens) / (time.time() - t)

valid, speeds, sample = 0, [], None
checked = min(5, len(data["val"]))
for row in data["val"].select(range(checked)):
    answer, speed = plan(row["messages"])
    speeds.append(speed)
    try:
        parsed = json.loads(answer)
        if "script" in parsed and "plan" in parsed and 5 <= len(parsed["script"]["scenes"]) <= 8:
            valid += 1
            sample = sample or parsed
    except Exception:
        pass

print(f"Valid JSON with script + plan: {valid}/{checked}. GPU speed {sum(speeds)/len(speeds):.0f} tokens/sec.")
if sample:
    print("Sample hook:", sample["script"]["hook"])
    for s in sample["script"]["scenes"]:
        print(" ", s["id"], s["purpose"], "|", s["narration"])

In [ ]:
# Hugging Face upload. Paste your WRITE token in the box. It is not saved in the notebook.
from getpass import getpass
from huggingface_hub import HfApi

HF_TOKEN = getpass("Hugging Face write token: ")
HF_USER = HfApi().whoami(token = HF_TOKEN)["name"]
HF_REPO = f"{HF_USER}/{HF_MODEL}"
print("Will upload to", HF_REPO)

In [ ]:
# Export a 4-bit GGUF file for llama.cpp and upload it. This step builds llama.cpp and takes 10 to 25 minutes.
model.push_to_hub_gguf(
    HF_REPO,
    tokenizer,
    quantization_method = ["q4_k_m"],
    token = HF_TOKEN,
)

In [ ]:
from huggingface_hub import list_repo_files
files = [f for f in list_repo_files(HF_REPO, token = HF_TOKEN) if f.endswith(".gguf")]

print("\n===== COPY EVERYTHING BELOW AND SEND IT BACK =====")
print(json.dumps({
    "model": f"https://huggingface.co/{HF_REPO}",
    "gguf_files": files,
    "base_model": BASE_MODEL,
    "train_examples": len(data["train"]),
    "epochs": EPOCHS,
    "train_minutes": round(train_minutes, 1),
    "train_loss": round(stats.training_loss, 3),
    "val_loss": round(eval_loss, 3),
    "val_valid_json": f"{valid}/{checked}",
}, indent = 2))